# Практичні роботи 4–7. Мініпроєкт «Порівняння класичних моделей класифікації»

**Курс:** Розпізнавання образів та класичне машинне навчання

**Студент:** Чирва Данііл

**Група:** ІН-42/1

**Дата виконання:** 12.11.2026

**Набір даних:** Heart Disease (UCI), файл `heart_disease.csv`

## Постановка задачі

**Що досліджуємо:** пацієнта, якого описано клінічними показниками.

**Ознаки (13):** age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal.

**Що передбачаємо:** змінну `target`. Класів два, тому це бінарна класифікація з розміченими даними.

**Позитивний клас:** 1, тобто ознаки серцевого захворювання є. Клас 0 означає, що ознак немає.

**Практична мета:** за показниками пацієнта вгадати, чи є в нього ознаки захворювання, і чесно порівняти кілька класичних моделей за однією й тією самою схемою крос-валідації.

**Основна метрика:** F1 для класу 1. Додатково дивлюся на Recall класу 1 (пропустити хворого гірше, ніж помилково направити здорового) та ROC AUC. Метрику вибрано до експериментів і потім не змінювалась.

**Застереження:** це навчальний проєкт. Його результати не є діагнозом і не можуть бути основою для клінічних рішень.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/3_course/Image_Recognition_and_Classical_Machine_Learning"
DATA_PATH = os.path.join(PROJECT_DIR, "data", "heart_disease.csv")
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
REPORTS_DIR = os.path.join(PROJECT_DIR, "reports")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("Файл даних існує:", os.path.exists(DATA_PATH))

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/3_course/Image_Recognition_and_Classical_Machine_Learning
Файл даних існує: True


## Практична робота 4. Дані, Pipeline і логістична регресія

### Первинний аудит даних

In [2]:
df = pd.read_csv(DATA_PATH)

print("Розмір таблиці:", df.shape)
display(df.head())
print()
print(df.dtypes)

Розмір таблиці: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0



age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca          float64
thal        float64
target        int64
dtype: object


In [3]:
print("Пропуски по стовпцях:")
print(df.isna().sum())
print("Усього пропусків:", df.isna().sum().sum())
print()
print("Розподіл класів (кількість):")
print(df["target"].value_counts())
print("Розподіл класів (частка):")
print(df["target"].value_counts(normalize=True).round(3))
print()
print("Допустимі значення target:", sorted(df["target"].unique()))

Пропуски по стовпцях:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64
Усього пропусків: 6

Розподіл класів (кількість):
target
0    164
1    139
Name: count, dtype: int64
Розподіл класів (частка):
target
0    0.541
1    0.459
Name: proportion, dtype: float64

Допустимі значення target: [np.int64(0), np.int64(1)]


In [4]:
numeric_columns = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_columns = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
target_column = "target"

print("Числові ознаки:", numeric_columns)
print("Категоріальні ознаки:", categorical_columns)
print("Цільова змінна:", target_column)

Числові ознаки: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Категоріальні ознаки: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
Цільова змінна: target


### Підсумок первинного аудиту

| Характеристика | Значення |
|---|---|
| Кількість об'єктів | 303 |
| Кількість вхідних ознак | 13 |
| Числові ознаки | age, trestbps, chol, thalach, oldpeak (5) |
| Категоріальні ознаки | sex, cp, fbs, restecg, exang, slope, ca, thal (8) |
| Пропуски | 6 (ca: 4, thal: 2) |
| Розподіл класів | 164 об'єкти класу 0 (54,1 %) і 139 класу 1 (45,9 %) |
| Допустимі значення target | 0 і 1 |
| Train / test | 242 / 61, стратифіковано, random_state = 42 |

Класи майже збалансовані, тому F1 і Recall тут не спотворені дисбалансом. Ознаки sex, cp, fbs, restecg, exang, slope, ca і thal закодовано числами, але це номери категорій, а не величини, тому вони йдуть через OneHotEncoder, а не через StandardScaler.

In [5]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate

X = df.drop(columns=[target_column])
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Клас 0 і 1 у y_train:", y_train.value_counts().to_dict())
print("Клас 0 і 1 у y_test:", y_test.value_counts().to_dict())

X_train: (242, 13) | X_test: (61, 13)
Клас 0 і 1 у y_train: {0: 131, 1: 111}
Клас 0 і 1 у y_test: {0: 33, 1: 28}


In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"F1": "f1", "ROC_AUC": "roc_auc", "RECALL": "recall"}

print(cv)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

def make_preprocessor(scaler):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler)
    ])
    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])

preprocessor = make_preprocessor(StandardScaler())
preprocessor_noscale = make_preprocessor("passthrough")
preprocessor_minmax = make_preprocessor(MinMaxScaler())

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000))
])
model_pipeline_noscale = Pipeline([
    ("preprocessor", preprocessor_noscale),
    ("classifier", LogisticRegression(max_iter=2000))
])

preprocessor.fit(X_train)
X_train_prepared = preprocessor.transform(X_train)
print("Форма X_train після препроцесингу:", X_train_prepared.shape)
print("Є пропуски після препроцесингу:", np.isnan(X_train_prepared).any())

Форма X_train після препроцесингу: (242, 28)
Є пропуски після препроцесингу: False


In [8]:
def run_cv(pipe):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, error_score=np.nan)
        except ValueError:
            nan5 = np.full(5, np.nan)
            res = {"test_F1": nan5, "test_ROC_AUC": nan5, "test_RECALL": nan5, "fit_time": nan5}
    row = {
        "CV F1": np.nanmean(res["test_F1"]),
        "F1 std": np.nanstd(res["test_F1"]),
        "ROC AUC": np.nanmean(res["test_ROC_AUC"]),
        "ROC AUC std": np.nanstd(res["test_ROC_AUC"]),
        "Recall": np.nanmean(res["test_RECALL"]),
        "Час навчання, с": np.nanmean(res["fit_time"]) if not np.isnan(res["fit_time"]).all() else np.nan,
        "Збійних фолдів": int(np.isnan(res["test_F1"]).sum())
    }
    return row, caught

### Вплив масштабування на логістичну регресію

In [9]:
rows = []
for name, pipe in [("Зі StandardScaler", model_pipeline), ("Без масштабування", model_pipeline_noscale)]:
    row, caught = run_cv(pipe)
    pipe.fit(X_train, y_train)
    n_iter = pipe.named_steps["classifier"].n_iter_[0]
    n_warn = sum(1 for w in caught if w.category.__name__ == "ConvergenceWarning")
    row.update({"Pipeline": name, "Ітерацій": n_iter, "ConvergenceWarning": n_warn})
    rows.append(row)

df_lr_scale = pd.DataFrame(rows).set_index("Pipeline")
display(df_lr_scale.round(4))

,CV F1,F1 std,ROC AUC,ROC AUC std,Recall,"Час навчання, с",Збійних фолдів,Ітерацій,ConvergenceWarning
Pipeline,,,,,,,,,
Зі StandardScaler,0.8245,0.0069,0.9025,0.0144,0.7834,0.0166,0,18,0
Без масштабування,0.8245,0.0069,0.9001,0.0151,0.7834,0.1162,0,1371,0


**Що видно з таблиці.** За якістю прогнозів масштабування майже нічого не дало: CV F1 (0,8245) і CV Recall (0,7834) однакові в обох варіантах, а ROC AUC трохи вищий зі StandardScaler (0,9025 проти 0,9001). Справжня різниця в оптимізації: зі StandardScaler solver зійшовся за 18 ітерацій, без нього за 1371, і навчання було приблизно вшестеро довшим. Попереджень про збіжність не було, бо 1371 менше за ліміт max_iter = 2000. Далі працюю з масштабованим варіантом: якість така сама, але навчання швидше, а коефіцієнти числових ознак можна порівнювати між собою.

### L1 і L2 регуляризація, пошук за GridSearchCV

In [10]:
from sklearn.model_selection import GridSearchCV

C_values = [0.01, 0.1, 1, 10, 100]
param_grid_lr = [
    {"classifier__penalty": ["l1"], "classifier__solver": ["liblinear"], "classifier__C": C_values},
    {"classifier__penalty": ["l2"], "classifier__solver": ["liblinear"], "classifier__C": C_values},
]

grid_lr = GridSearchCV(model_pipeline, param_grid_lr, scoring="f1", cv=cv)
grid_lr.fit(X_train, y_train)

res_lr = pd.DataFrame(grid_lr.cv_results_)[["param_classifier__penalty", "param_classifier__C", "mean_test_score", "std_test_score"]]
res_lr.columns = ["penalty", "C", "CV F1", "F1 std"]
res_lr["C"] = res_lr["C"].astype(float)
display(res_lr.sort_values("CV F1", ascending=False).round(4))

best_f1_lr = res_lr["CV F1"].round(4).max()
ties_lr = res_lr[res_lr["CV F1"].round(4) == best_f1_lr]
print("Конфігурацій з найкращим F1:")
display(ties_lr.round(4))
best_row_lr = ties_lr.sort_values(["C", "penalty"], ascending=[True, False]).iloc[0]
best_penalty, best_C = best_row_lr["penalty"], best_row_lr["C"]
print("Обрано: penalty =", best_penalty, ", C =", best_C, ", CV F1 =", round(best_row_lr["CV F1"], 4), ", std =", round(best_row_lr["F1 std"], 4))

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead o

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead o

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead o

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logi

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logi

,penalty,C,CV F1,F1 std
9,l2,100.00,0.8363,0.0127
4,l1,100.00,0.8363,0.0127
8,l2,10.00,0.8363,0.0127
3,l1,10.00,0.8321,0.0108
7,l2,1.00,0.8245,0.0069
6,l2,0.10,0.8140,0.0299
2,l1,1.00,0.8123,0.0173
1,l1,0.10,0.7926,0.0306
5,l2,0.01,0.7658,0.0171
0,l1,0.01,0.0000,0.0000


Конфігурацій з найкращим F1:


,penalty,C,CV F1,F1 std
4,l1,100.0,0.8363,0.0127
8,l2,10.0,0.8363,0.0127
9,l2,100.0,0.8363,0.0127


Обрано: penalty = l2 , C = 10.0 , CV F1 = 0.8363 , std = 0.0127


In [11]:
plt.figure(figsize=(8, 5))
for pen in ["l1", "l2"]:
    part = res_lr[res_lr["penalty"] == pen]
    plt.plot(part["C"], part["CV F1"], marker="o", label=pen.upper())
plt.xscale("log")
plt.xlabel("C (логарифмічна шкала)")
plt.ylabel("Середній CV F1")
plt.title("Логістична регресія: CV F1 залежно від C")
plt.legend()
plt.grid(True)
plt.show()

**Вплив C і типу штрафу.**

- При C = 0,01 штраф занадто сильний: для L2 CV F1 лише 0,7658, а для L1 він нульовий, тобто позитивний клас фактично не передбачався.
- З ростом C якість зростає й виходить на плато. L2: 0,7658 → 0,8140 → 0,8245 → 0,8363 (C = 10) → 0,8363 (C = 100). L1: 0 → 0,7926 → 0,8123 → 0,8321 → 0,8363 (C = 100).
- При малих C L2 помітно кращий за L1, а від C = 10 різниця зникає.
- Три конфігурації дали однаковий CV F1 = 0,8363 з однаковим std = 0,0127: L2 з C = 10, L2 з C = 100 і L1 з C = 100. Тому `best_params_` тут нічого не вирішує, і я заздалегідь задав правило: за однакового F1 беру менше C (сильніша регуляризація, простіша модель), а при рівних C беру L2.

**Обрано: L2, C = 10, solver = liblinear, зі StandardScaler. CV F1 = 0,8363, std = 0,0127.**

In [12]:
best_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(penalty=best_penalty, C=best_C, solver="liblinear", max_iter=2000))
])
best_lr.fit(X_train, y_train)

coef_df = pd.DataFrame({
    "feature": best_lr.named_steps["preprocessor"].get_feature_names_out(),
    "coef": best_lr.named_steps["classifier"].coef_[0]
})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_sorted = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)

print("П'ять найбільших за модулем коефіцієнтів:")
display(coef_sorted.head(5).round(4))

П'ять найбільших за модулем коефіцієнтів:


/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,feature,coef,abs_coef
0,cat__ca_0.0,-1.9262,1.9262
1,cat__cp_4.0,1.1886,1.1886
2,cat__thal_7.0,0.9779,0.9779
3,cat__cp_1.0,-0.8921,0.8921
4,cat__sex_0.0,-0.8802,0.8802


**Як читати коефіцієнти.** Додатний коефіцієнт підвищує log-odds класу 1 за інших рівних умов, від'ємний знижує. Найбільший внесок мають `ca = 0` (−1,9262, тобто немає уражених судин, що знижує шанс класу 1), `cp = 4` (1,1886), `thal = 7` (0,9779), `cp = 1` (−0,8921) і `sex = 0` (−0,8802). Це зв'язки в даних, а не доказ причини чи медичний висновок.

### Додаткове завдання: усі коефіцієнти логістичної регресії

In [13]:
print("Усі коефіцієнти (", len(coef_sorted), "), відсортовані за модулем:")
display(coef_sorted.round(4))

plt.figure(figsize=(8, 8))
coef_rev = coef_sorted.iloc[::-1]
plt.barh(coef_rev["feature"], coef_rev["coef"])
plt.axvline(0, color="black")
plt.xlabel("Коефіцієнт")
plt.title("Коефіцієнти логістичної регресії")
plt.tight_layout()
plt.show()

Усі коефіцієнти ( 28 ), відсортовані за модулем:


,feature,coef,abs_coef
0,cat__ca_0.0,-1.9262,1.9262
1,cat__cp_4.0,1.1886,1.1886
2,cat__thal_7.0,0.9779,0.9779
3,cat__cp_1.0,-0.8921,0.8921
4,cat__sex_0.0,-0.8802,0.8802
5,cat__ca_2.0,0.8409,0.8409
6,cat__sex_1.0,0.8337,0.8337
7,cat__ca_3.0,0.8003,0.8003
8,cat__slope_2.0,0.7799,0.7799
9,cat__cp_3.0,-0.6662,0.6662


**Що додає повна картина.** Із 28 коефіцієнтів перші 13 за модулем належать one-hot колонкам, а перша числова ознака `trestbps` (0,4302) лише 14-та. Числові ознаки загалом впливають слабше: `age` має −0,1122, `chol` 0,2118. Найслабший коефіцієнт у `restecg = 0` (−0,0472).

Є ще дві обережності. По-перше, дві колонки однієї категоріальної ознаки часто мають протилежні знаки й схожий модуль (наприклад, `sex = 0` дає −0,8802, а `sex = 1` дає 0,8337), тому їх треба читати разом, а не як дві незалежні ознаки. По-друге, ознаки можуть бути залежними між собою, тому окремий коефіцієнт не показує чистий вплив однієї ознаки.

### Висновок за практичною роботою 4

- **Масштабування.** На якість логістичної регресії воно майже не вплинуло (CV F1 0,8245 в обох випадках), але прискорило збіжність: 18 ітерацій проти 1371.
- **Регуляризація.** Дуже мале C шкодить (при C = 0,01 L2 дає 0,7658, а L1 зовсім ламається). При C від 10 обидва штрафи дають однаковий результат 0,8363.
- **Вибір.** L2, C = 10, бо за рівного F1 простіша модель краща. CV F1 = 0,8363, std = 0,0127.
- **Коефіцієнти.** Найсильніші `ca = 0`, `cp = 4`, `thal = 7`. Це кореляції в даних, не причинність.
- X_test і y_test на цьому етапі не використовувались.

## Практична робота 5. k-NN та ймовірнісні моделі

### k-NN: вплив масштабування (k = 5), StandardScaler, MinMaxScaler і без масштабування

In [14]:
from sklearn.neighbors import KNeighborsClassifier

def make_knn(prep, k=5, p=2):
    return Pipeline([
        ("preprocessor", prep),
        ("classifier", KNeighborsClassifier(n_neighbors=k, metric="minkowski", p=p))
    ])

rows = []
for name, prep in [("StandardScaler", preprocessor), ("MinMaxScaler", preprocessor_minmax), ("Без масштабування", preprocessor_noscale)]:
    row, _ = run_cv(make_knn(prep))
    row["Варіант"] = name
    rows.append(row)

df_knn_scale = pd.DataFrame(rows).set_index("Варіант")
display(df_knn_scale.round(4))

,CV F1,F1 std,ROC AUC,ROC AUC std,Recall,"Час навчання, с",Збійних фолдів
Варіант,,,,,,,
StandardScaler,0.7989,0.0426,0.8712,0.0387,0.7826,0.0118,0
MinMaxScaler,0.8008,0.0481,0.8806,0.0336,0.7921,0.0115,0
Без масштабування,0.5120,0.0134,0.6419,0.0319,0.4866,0.0113,0


**Що видно.** Для k-NN масштаб вирішальний. Без масштабування CV F1 падає з 0,7989 до 0,5120, ROC AUC з 0,8712 до 0,6419, а Recall з 0,7826 до 0,4866, тобто модель пропускає більше половини хворих. AUC 0,64 майже не відрізняється від випадкового вгадування.

**Чому так.** k-NN рахує відстань по всіх ознаках одночасно. Без масштабування chol, trestbps і thalach мають діапазон у сотні одиниць, oldpeak в одиниці, а one-hot колонки лише 0 або 1. Тому сусідів визначають переважно перші три ознаки, а cp, ca і thal майже не впливають. Невеликий std без масштабування (0,0134) означає лише стабільно погану модель.

**StandardScaler проти MinMaxScaler.** При k = 5 вони близькі: F1 0,7989 і 0,8008, AUC 0,8712 і 0,8806. Різниця менша за розкид між фолдами (0,043 і 0,048), тому на цьому кроці сказати, що один краще, не можна. Для логістичної регресії масштаб майже нічого не змінював, бо вона підлаштовує коефіцієнти під шкалу, а k-NN так не вміє.

### k-NN: вибір k (StandardScaler, евклідова відстань)

In [15]:
k_list = [3, 5, 7, 9, 11, 15]
rows = []
for k in k_list:
    row, _ = run_cv(make_knn(preprocessor, k=k))
    row["k"] = k
    rows.append(row)

df_knn_k = pd.DataFrame(rows).set_index("k")
display(df_knn_k.round(4))

best_f1_k = df_knn_k["CV F1"].round(4).max()
k_best = df_knn_k[df_knn_k["CV F1"].round(4) == best_f1_k].index.max()
print("Найкраще k:", k_best, "| CV F1:", round(best_f1_k, 4))

plt.figure(figsize=(8, 5))
plt.plot(df_knn_k.index, df_knn_k["CV F1"], marker="o")
plt.xlabel("k (кількість сусідів)")
plt.ylabel("Середній CV F1")
plt.title("k-NN: CV F1 залежно від k")
plt.grid(True)
plt.show()

,CV F1,F1 std,ROC AUC,ROC AUC std,Recall,"Час навчання, с",Збійних фолдів
k,,,,,,,
3,0.8042,0.0272,0.8709,0.0406,0.8008,0.0138,0
5,0.7989,0.0426,0.8712,0.0387,0.7826,0.0137,0
7,0.7942,0.0184,0.8721,0.0306,0.7656,0.0118,0
9,0.7872,0.0294,0.8744,0.0210,0.7557,0.0117,0
11,0.7791,0.0273,0.8806,0.0252,0.7375,0.0113,0
15,0.7651,0.0246,0.8729,0.0202,0.7209,0.0125,0


Найкраще k: 3 | CV F1: 0.8042


**Що видно.** З ростом k CV F1 поступово падає: 0,8042 (k = 3), 0,7989 (5), 0,7942 (7), 0,7872 (9), 0,7791 (11), 0,7651 (15). Recall падає ще помітніше, з 0,8008 до 0,7209. Велике k згладжує межу й втрачає локальні деталі, а мале k точніше відстежує локальну структуру. Найкраще за F1 тут k = 3. Значення k = 1 і 2 я не перевіряв, лише набір 3, 5, 7, 9, 11, 15.

### k-NN: евклідова і мангеттенська відстані

In [16]:
rows = []
for name, p in [("Евклідова (p=2)", 2), ("Мангеттенська (p=1)", 1)]:
    row, _ = run_cv(make_knn(preprocessor, k=k_best, p=p))
    row["Міра відстані"] = name
    rows.append(row)

df_knn_dist = pd.DataFrame(rows).set_index("Міра відстані")
print("k =", k_best)
display(df_knn_dist.round(4))

k = 3


,CV F1,F1 std,ROC AUC,ROC AUC std,Recall,"Час навчання, с",Збійних фолдів
Міра відстані,,,,,,,
Евклідова (p=2),0.8042,0.0272,0.8709,0.0406,0.8008,0.0112,0
Мангеттенська (p=1),0.8175,0.0304,0.8768,0.0333,0.7917,0.0100,0


**Що видно.** Для k = 3 мангеттенська відстань дала CV F1 0,8175, евклідова 0,8042. Різниця 0,0133 менша за std між фолдами (0,027 і 0,030), тому це лише слабка тенденція. Можливе пояснення: в one-hot просторі мангеттенська відстань менш чутлива до окремих великих відхилень.

### Додаткове завдання: MinMaxScaler і інші міри Мінковського

In [17]:
p_list = [1, 1.5, 2, 3, 4]
scalers = [("StandardScaler", preprocessor), ("MinMaxScaler", preprocessor_minmax)]

rows = []
for scaler_name, prep in scalers:
    for p in p_list:
        for k in k_list:
            row, _ = run_cv(make_knn(prep, k=k, p=p))
            row.update({"Масштаб": scaler_name, "p": p, "k": k})
            rows.append(row)

df_knn_all = pd.DataFrame(rows)

summary = df_knn_all.loc[df_knn_all.groupby(["Масштаб", "p"])["CV F1"].idxmax()]
print("Найкращий k для кожної пари (масштабувальник, p):")
display(summary[["Масштаб", "p", "k", "CV F1", "F1 std", "ROC AUC", "Recall"]].round(4).reset_index(drop=True))

plt.figure(figsize=(8, 5))
for scaler_name, _ in scalers:
    part = summary[summary["Масштаб"] == scaler_name]
    plt.plot(part["p"], part["CV F1"], marker="o", label=scaler_name)
plt.xlabel("p (параметр відстані Мінковського)")
plt.ylabel("Найкращий CV F1 по k")
plt.title("k-NN: масштабувальник і міра Мінковського")
plt.legend()
plt.grid(True)
plt.show()

best_f1_knn = df_knn_all["CV F1"].round(4).max()
ties_knn = df_knn_all[df_knn_all["CV F1"].round(4) == best_f1_knn]
best_knn_row = ties_knn.sort_values(["k", "p"], ascending=[False, True]).iloc[0]
knn_scaler_name, knn_p, knn_k = best_knn_row["Масштаб"], best_knn_row["p"], int(best_knn_row["k"])
knn_prep = dict(scalers)[knn_scaler_name]
print("Найкраща k-NN:", knn_scaler_name, ", p =", knn_p, ", k =", knn_k, ", CV F1 =", round(best_knn_row["CV F1"], 4), ", std =", round(best_knn_row["F1 std"], 4))
print("Кількість конфігурацій з таким F1:", len(ties_knn))

Найкращий k для кожної пари (масштабувальник, p):


,Масштаб,p,k,CV F1,F1 std,ROC AUC,Recall
0,MinMaxScaler,1.0,11,0.8291,0.0375,0.8971,0.8190
1,MinMaxScaler,1.5,11,0.8291,0.0375,0.9024,0.8190
2,MinMaxScaler,2.0,11,0.8182,0.0417,0.9035,0.8012
3,MinMaxScaler,3.0,11,0.8189,0.0473,0.8982,0.8099
4,MinMaxScaler,4.0,11,0.8226,0.0520,0.8984,0.8099
5,StandardScaler,1.0,7,0.8244,0.0484,0.8913,0.8008
6,StandardScaler,1.5,5,0.8172,0.0474,0.8890,0.7826
7,StandardScaler,2.0,3,0.8042,0.0272,0.8709,0.8008
8,StandardScaler,3.0,5,0.7866,0.0361,0.8607,0.7735
9,StandardScaler,4.0,5,0.7564,0.0103,0.8387,0.7292


Найкраща k-NN: MinMaxScaler , p = 1.0 , k = 11 , CV F1 = 0.8291 , std = 0.0375
Кількість конфігурацій з таким F1: 2


**Що додали MinMaxScaler і відстані Мінковського.** Я перебрав усі комбінації: масштабувальник (Standard, MinMax) x p = 1; 1,5; 2; 3; 4 x k = 3, 5, 7, 9, 11, 15, усе лише за CV на train. Параметр p керує відстанню: p = 1 мангеттенська, p = 2 евклідова, більше p сильніше підкреслює найбільшу різницю за однією ознакою.

- **StandardScaler** добре працює з малими p і гіршає зі зростанням: найкращий F1 0,8244 (p = 1, k = 7), 0,8172 (p = 1,5), 0,8042 (p = 2), 0,7866 (p = 3) і 0,7564 (p = 4).
- **MinMaxScaler** стабільніший: найкращий F1 від 0,8182 до 0,8291 при всіх p, і найкраще k скрізь дорівнює 11.
- Менше p загалом краще, а MinMax менш чутливий до вибору p. Імовірна причина: після MinMax усі числові ознаки лежать у [0, 1], як і one-hot колонки, тому відстань рівномірніше враховує ознаки обох типів.
- Усі відмінності між найкращими конфігураціями (0,818 до 0,829) менші за std між фолдами (0,037 до 0,052), тобто це різниця в межах шуму.

**Найкраща k-NN: MinMaxScaler, p = 1 (манхеттен), k = 11, CV F1 = 0,8291, std = 0,0375, Recall = 0,8190.** Таке саме значення F1 дало й p = 1,5, але за правилом (при рівності менше p) беру p = 1.

Обмеження: цю конфігурацію вибрано з 60 варіантів на тих самих фолдах, тому її CV F1 може бути трохи оптимістичним.

### GaussianNB, LDA і QDA

In [18]:
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

prob_models = [
    ("GaussianNB", GaussianNB()),
    ("LDA", LinearDiscriminantAnalysis()),
    ("QDA (reg_param=0)", QuadraticDiscriminantAnalysis())
]

rows = []
for name, model in prob_models:
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", model)])
    row, caught = run_cv(pipe)
    row["Модель"] = name
    row["Попереджень"] = len(caught)
    rows.append(row)

df_prob = pd.DataFrame(rows).set_index("Модель")
display(df_prob.round(4))

/tmp/ipykernel_149/925694927.py:10: RuntimeWarning: Mean of empty slice
  "CV F1": np.nanmean(res["test_F1"]),
/usr/local/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_149/925694927.py:12: RuntimeWarning: Mean of empty slice
  "ROC AUC": np.nanmean(res["test_ROC_AUC"]),
/tmp/ipykernel_149/925694927.py:14: RuntimeWarning: Mean of empty slice
  "Recall": np.nanmean(res["test_RECALL"]),


,CV F1,F1 std,ROC AUC,ROC AUC std,Recall,"Час навчання, с",Збійних фолдів,Попереджень
Модель,,,,,,,,
GaussianNB,0.7495,0.0215,0.8773,0.0214,0.8458,0.0111,0,0
LDA,0.8128,0.0257,0.8993,0.0156,0.7656,0.0126,0,0
QDA (reg_param=0),NaN,NaN,NaN,NaN,NaN,NaN,5,0


### Описовий аналіз для GaussianNB: розподіли числових ознак за класами

In [19]:
train_df = X_train.copy()
train_df[target_column] = y_train

for col in numeric_columns:
    plt.figure(figsize=(7, 4))
    plt.hist(train_df[train_df[target_column] == 0][col], bins=15, alpha=0.6, label="клас 0")
    plt.hist(train_df[train_df[target_column] == 1][col], bins=15, alpha=0.6, label="клас 1")
    plt.xlabel(col)
    plt.ylabel("Кількість")
    plt.title("Розподіл " + col + " за класами (train)")
    plt.legend()
    plt.show()

**Описовий аналіз для GaussianNB (лише train).** Класи найкраще розрізняють `thalach` і `oldpeak`: середній thalach у класі 0 дорівнює 158,5, а в класі 1 139,9, середній oldpeak 0,6 проти 1,5. Для `age` різниця менша (52,8 проти 56,6), а `chol` і `trestbps` майже не розрізняють класи (246,1 проти 254,3 і 128,8 проти 133,6) і мають сильне перекриття. Крім того, `oldpeak` має розподіл, скошений вправо (мінімум 0, довгий правий хвіст), тому припущення GaussianNB про нормальність для неї порушується. Це спостереження для пояснення результатів, а не підстава міняти модель.

### QDA: стабільність і reg_param

In [20]:
reg_grid = [0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1]

preprocessor_num_only = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_columns)
])

rows = []
for set_name, prep in [("Усі 13 ознак", preprocessor), ("Лише 5 числових", preprocessor_num_only)]:
    for reg in reg_grid:
        pipe = Pipeline([("preprocessor", prep), ("classifier", QuadraticDiscriminantAnalysis(reg_param=reg))])
        row, caught = run_cv(pipe)
        row.update({"Набір ознак": set_name, "reg_param": reg, "Попереджень": len(caught)})
        rows.append(row)

df_qda = pd.DataFrame(rows)
display(df_qda[["Набір ознак", "reg_param", "CV F1", "F1 std", "ROC AUC", "Recall", "Збійних фолдів", "Попереджень"]].round(4))

qda_all = df_qda[df_qda["Набір ознак"] == "Усі 13 ознак"]
best_f1_qda = qda_all["CV F1"].round(4).max()
best_reg = qda_all[qda_all["CV F1"].round(4) == best_f1_qda]["reg_param"].min()
print("Найкращий reg_param (усі ознаки):", best_reg, "| CV F1:", round(best_f1_qda, 4))

/tmp/ipykernel_149/925694927.py:10: RuntimeWarning: Mean of empty slice
  "CV F1": np.nanmean(res["test_F1"]),
/usr/local/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_149/925694927.py:12: RuntimeWarning: Mean of empty slice
  "ROC AUC": np.nanmean(res["test_ROC_AUC"]),
/tmp/ipykernel_149/925694927.py:14: RuntimeWarning: Mean of empty slice
  "Recall": np.nanmean(res["test_RECALL"]),


,Набір ознак,reg_param,CV F1,F1 std,ROC AUC,Recall,Збійних фолдів,Попереджень
0,Усі 13 ознак,0.000,NaN,NaN,NaN,NaN,5,0
1,Усі 13 ознак,0.001,0.7543,0.0509,0.8271,0.7739,0,0
2,Усі 13 ознак,0.010,0.7709,0.0440,0.8343,0.7743,0,0
3,Усі 13 ознак,0.050,0.7837,0.0343,0.8619,0.7834,0,0
4,Усі 13 ознак,0.100,0.7962,0.0271,0.8680,0.7743,0,0
5,Усі 13 ознак,0.200,0.7957,0.0274,0.8778,0.7561,0,0
6,Усі 13 ознак,0.300,0.7921,0.0322,0.8825,0.7561,0,0
7,Усі 13 ознак,0.500,0.7988,0.0286,0.8879,0.7561,0,0
8,Усі 13 ознак,0.700,0.8022,0.0451,0.8848,0.7561,0,0
9,Усі 13 ознак,1.000,0.7717,0.0447,0.8718,0.7383,0,0


Найкращий reg_param (усі ознаки): 0.7 | CV F1: 0.8022


**Стабільність QDA.** При reg_param = 0 QDA не навчилась на жодному із 5 фолдів: у моїй версії scikit-learn вона кидає помилку, що коваріаційна матриця класу не має повного рангу (в one-hot ознаках є лінійні залежності, а об'єктів у фолді небагато), тому для цього рядка метрик немає. Це і є нестабільність.

Невелика регуляризація все виправляє. Для всіх 13 ознак CV F1 такий: 0,7543 (reg_param = 0,001), 0,7709 (0,01), 0,7837 (0,05), 0,7962 (0,1; std 0,0271), 0,8022 (0,7; std 0,0451), 0,7717 (1). Загальна тенденція така: після 0,1 крива майже пласка в межах шуму, тож 0,7 не набагато краще за 0,1. Лише з 5 числових ознак QDA працює погано (CV F1 близько 0,60 при будь-якому reg_param), тобто категоріальні ознаки несуть багато корисної інформації. Найкращий reg_param серед усіх ознак: 0,7.

### Позавибіркові ймовірності та калібрування

In [21]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

oof_models = {
    "k-NN": make_knn(knn_prep, k=knn_k, p=knn_p),
    "GaussianNB": Pipeline([("preprocessor", preprocessor), ("classifier", GaussianNB())]),
    "LDA": Pipeline([("preprocessor", preprocessor), ("classifier", LinearDiscriminantAnalysis())]),
    "QDA": Pipeline([("preprocessor", preprocessor), ("classifier", QuadraticDiscriminantAnalysis(reg_param=best_reg))]),
}

oof_proba = {}
for name, pipe in oof_models.items():
    oof_proba[name] = cross_val_predict(pipe, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

df_oof = pd.DataFrame(oof_proba, index=X_train.index)
df_oof["y_true"] = y_train
print("Перші 8 об'єктів train, ймовірність класу 1 від моделі, яка їх не бачила:")
display(df_oof.head(8).round(4))

base_brier = brier_score_loss(y_train, np.full(len(y_train), y_train.mean()))
rows = [{"Модель": name, "Brier": brier_score_loss(y_train, pr)} for name, pr in oof_proba.items()]
df_brier = pd.DataFrame(rows).set_index("Модель")
print("Brier базової лінії:", round(base_brier, 4))
display(df_brier.round(4))

plt.figure(figsize=(6, 6))
for name, pr in oof_proba.items():
    frac_pos, mean_pred = calibration_curve(y_train, pr, n_bins=5)
    plt.plot(mean_pred, frac_pos, marker="o", label=name)
plt.plot([0, 1], [0, 1], "--", color="gray", label="ідеально")
plt.xlabel("Середня передбачена ймовірність")
plt.ylabel("Реальна частка класу 1")
plt.title("Калібрування (позавибіркові прогнози на train)")
plt.legend()
plt.grid(True)
plt.show()

Перші 8 об'єктів train, ймовірність класу 1 від моделі, яка їх не бачила:


,k-NN,GaussianNB,LDA,QDA,y_true
180,0.7273,1.0000,0.9061,0.4873,1
208,0.0909,0.0000,0.0212,0.0598,0
167,0.2727,0.0000,0.4727,0.2452,0
105,0.1818,0.4342,0.1587,0.1639,0
297,0.6364,1.0000,0.6183,0.7661,1
120,0.9091,1.0000,0.9795,0.7349,1
55,1.0000,1.0000,0.9982,0.9945,1
123,0.8182,1.0000,0.9513,0.9995,1


Brier базової лінії: 0.2483


,Brier
Модель,
k-NN,0.1243
GaussianNB,0.2367
LDA,0.1276
QDA,0.1379


**Що показують ймовірності.** Позавибіркова ймовірність означає, що її видала модель, яка цей об'єкт під час навчання не бачила. Brier score (менше краще, базова лінія 0,2483): k-NN 0,1243, LDA 0,1276, QDA 0,1379, GaussianNB 0,2367. Тобто GaussianNB майже не кращий за наївне вгадування частки класу, і в першому ж зрізі з 8 об'єктів сім отримали ймовірність рівно 0 або 1. Причина: GaussianNB вважає ознаки незалежними, а one-hot колонки однієї ознаки залежні, тому докази «множаться» і модель стає надто самовпевненою. За Brier найкраще відкалібровані k-NN і LDA.

Про GaussianNB варто додати: його Recall найвищий серед ймовірнісних моделей (0,8458), але ціною низької точності, тому F1 лише 0,7495.

In [22]:
prob_rows = []
for name, model in [("GaussianNB", GaussianNB()), ("LDA", LinearDiscriminantAnalysis()), ("QDA", QuadraticDiscriminantAnalysis(reg_param=best_reg))]:
    row, _ = run_cv(Pipeline([("preprocessor", preprocessor), ("classifier", model)]))
    row["Модель"] = name
    prob_rows.append(row)
df_prob_best = pd.DataFrame(prob_rows).set_index("Модель")
display(df_prob_best.round(4))

prob_best_name = df_prob_best["CV F1"].idxmax()
print("Найкраща ймовірнісна модель:", prob_best_name)

,CV F1,F1 std,ROC AUC,ROC AUC std,Recall,"Час навчання, с",Збійних фолдів
Модель,,,,,,,
GaussianNB,0.7495,0.0215,0.8773,0.0214,0.8458,0.0118,0
LDA,0.8128,0.0257,0.8993,0.0156,0.7656,0.0116,0
QDA,0.8022,0.0451,0.8848,0.0399,0.7561,0.0110,0


Найкраща ймовірнісна модель: LDA


### Висновок за практичною роботою 5

- **Масштаб для k-NN.** Критичний: без нього CV F1 0,5120, зі StandardScaler 0,7989, з MinMaxScaler 0,8008.
- **k і відстань.** За чистою евклідовою схемою найкраще k = 3 (F1 0,8042), мангеттенська відстань дала 0,8175. У повному переборі найкращою стала конфігурація MinMaxScaler, p = 1, k = 11 (CV F1 0,8291). Відмінності між топ-конфігураціями менші за розкид між фолдами.
- **Ймовірнісні моделі.** Найкраща LDA: CV F1 0,8128, ROC AUC 0,8993, std 0,0257. GaussianNB має найменший std (0,0215), але найнижчий F1 (0,7495) і найгірші ймовірності. QDA нестабільна при reg_param = 0 і прийнятна при reg_param = 0,7 (F1 0,8022, але std 0,0451).
- **Кандидати.** k-NN: MinMaxScaler, p = 1, k = 11. Ймовірнісна модель: LDA.
- Test у цій частині не використовувався.

## Практична робота 6. SVM

### Лінійна SVM

In [23]:
from sklearn.svm import SVC

C_svc = [0.1, 1, 10]
rows = []
for C in C_svc:
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", SVC(kernel="linear", C=C))])
    row, _ = run_cv(pipe)
    row["C"] = C
    rows.append(row)

df_svm_lin = pd.DataFrame(rows).set_index("C")
display(df_svm_lin.round(4))

,CV F1,F1 std,ROC AUC,ROC AUC std,Recall,"Час навчання, с",Збійних фолдів
C,,,,,,,
0.1,0.7962,0.0536,0.9047,0.0195,0.7375,0.0120,0
1.0,0.8025,0.0309,0.8980,0.0149,0.7747,0.0121,0
10.0,0.8000,0.0263,0.8968,0.0109,0.7747,0.0153,0


### SVM з RBF-ядром

In [24]:
gamma_list = ["scale", 0.01, 0.1]
rows = []
for C in C_svc:
    for g in gamma_list:
        pipe = Pipeline([("preprocessor", preprocessor), ("classifier", SVC(kernel="rbf", C=C, gamma=g))])
        row, _ = run_cv(pipe)
        row.update({"C": C, "gamma": g})
        rows.append(row)

df_svm_rbf = pd.DataFrame(rows)
display(df_svm_rbf[["C", "gamma", "CV F1", "F1 std", "ROC AUC", "Recall"]].round(4))

,C,gamma,CV F1,F1 std,ROC AUC,Recall
0,0.1,scale,0.7630,0.0159,0.8855,0.6937
1,0.1,0.01,0.2461,0.1129,0.8821,0.1451
2,0.1,0.1,0.7652,0.0174,0.8848,0.7028
3,1.0,scale,0.7893,0.0271,0.8884,0.7648
4,1.0,0.01,0.7951,0.0292,0.8957,0.7561
5,1.0,0.1,0.7893,0.0271,0.8867,0.7648
6,10.0,scale,0.7480,0.0228,0.8229,0.7470
7,10.0,0.01,0.8270,0.0330,0.8968,0.7830
8,10.0,0.1,0.7558,0.0328,0.8230,0.7652


### GridSearchCV для SVM

In [25]:
svm_pipe = Pipeline([("preprocessor", preprocessor), ("classifier", SVC())])
param_grid_svm = [
    {"classifier__kernel": ["linear"], "classifier__C": [0.1, 1, 10]},
    {"classifier__kernel": ["rbf"], "classifier__C": [0.1, 1, 10], "classifier__gamma": ["scale", 0.01, 0.1]},
]

grid_svm = GridSearchCV(svm_pipe, param_grid_svm, scoring="f1", cv=cv)
grid_svm.fit(X_train, y_train)

res_svm = pd.DataFrame(grid_svm.cv_results_)[["param_classifier__kernel", "param_classifier__C", "param_classifier__gamma", "mean_test_score", "std_test_score"]]
res_svm.columns = ["kernel", "C", "gamma", "CV F1", "F1 std"]
display(res_svm.sort_values("CV F1", ascending=False).round(4))

best_f1_svm = res_svm["CV F1"].round(4).max()
ties_svm = res_svm[res_svm["CV F1"].round(4) == best_f1_svm]
best_row_svm = ties_svm.sort_values(["C", "kernel"], ascending=[True, True]).iloc[0]
svm_kernel, svm_C, svm_gamma = best_row_svm["kernel"], best_row_svm["C"], best_row_svm["gamma"]
print("Конфігурацій з найкращим F1:", len(ties_svm))
print("Обрано: kernel =", svm_kernel, ", C =", svm_C, ", gamma =", svm_gamma, ", CV F1 =", round(best_row_svm["CV F1"], 4), ", std =", round(best_row_svm["F1 std"], 4))
print("best_params_ з GridSearchCV:", grid_svm.best_params_)

,kernel,C,gamma,CV F1,F1 std
10,rbf,10.0,0.01,0.8270,0.0330
1,linear,1.0,NaN,0.8025,0.0309
2,linear,10.0,NaN,0.8000,0.0263
0,linear,0.1,NaN,0.7962,0.0536
7,rbf,1.0,0.01,0.7951,0.0292
8,rbf,1.0,0.1,0.7893,0.0271
6,rbf,1.0,scale,0.7893,0.0271
5,rbf,0.1,0.1,0.7652,0.0174
3,rbf,0.1,scale,0.7630,0.0159
11,rbf,10.0,0.1,0.7558,0.0328


Конфігурацій з найкращим F1: 1
Обрано: kernel = rbf , C = 10.0 , gamma = 0.01 , CV F1 = 0.827 , std = 0.033
best_params_ з GridSearchCV: {'classifier__C': 10, 'classifier__gamma': 0.01, 'classifier__kernel': 'rbf'}


### Висновок за практичною роботою 6

**Лінійна SVM** майже не залежить від C: F1 0,7962 (C = 0,1), 0,8025 (C = 1), 0,8000 (C = 10). Різниця 0,006 набагато менша за std між фолдами (0,026 до 0,054), тож це в межах шуму, хоча при C = 0,1 розкид між фолдами найбільший.

**RBF-SVM** дуже чутлива до параметрів. При C = 0,1 і gamma = 0,01 модель недонавчена: F1 лише 0,2461 і Recall 0,1451, бо межа надто гладка й штраф за помилки малий. Найкращою виявилась комбінація C = 10, gamma = 0,01: CV F1 0,8270 (std 0,0330), ROC AUC 0,8968, Recall 0,7830. Для великих gamma (0,1 і «scale») при C = 10 якість падає до 0,75, що схоже на перенавчання.

**GridSearchCV** перебрав 12 конфігурацій (3 лінійні, 9 з RBF), переможець єдиний, і `best_params_` збігається з моєю вибіркою: RBF, C = 10, gamma = 0,01. Обмеження: C = 10 лежить на краю сітки, тому більші значення C я не перевіряв.

Test не використовувався.

## Практична робота 7. Порівняння моделей і вибір фінальної

In [26]:
from sklearn.base import clone

svm_kwargs = {"kernel": svm_kernel, "C": svm_C}
if svm_kernel == "rbf":
    svm_kwargs["gamma"] = svm_gamma

candidates = {
    "Logistic Regression": best_lr,
    "k-NN": make_knn(knn_prep, k=knn_k, p=knn_p),
    "GaussianNB": Pipeline([("preprocessor", preprocessor), ("classifier", GaussianNB())]),
    "LDA": Pipeline([("preprocessor", preprocessor), ("classifier", LinearDiscriminantAnalysis())]),
    "QDA": Pipeline([("preprocessor", preprocessor), ("classifier", QuadraticDiscriminantAnalysis(reg_param=best_reg))]),
    "SVM": Pipeline([("preprocessor", preprocessor), ("classifier", SVC(**svm_kwargs))]),
}

candidate_params = {
    "Logistic Regression": f"penalty={best_penalty}, C={best_C}, solver=liblinear",
    "k-NN": f"{knn_scaler_name}, k={knn_k}, p={knn_p}",
    "GaussianNB": "-",
    "LDA": "-",
    "QDA": f"reg_param={best_reg}",
    "SVM": ", ".join(f"{a}={b}" for a, b in svm_kwargs.items()),
}

def oof_scores(pipe):
    if hasattr(pipe, "predict_proba"):
        return cross_val_predict(pipe, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    return cross_val_predict(pipe, X_train, y_train, cv=cv, method="decision_function")

cv_scores = {name: oof_scores(pipe) for name, pipe in candidates.items()}
print("Кандидати:", list(candidates))

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logi

Кандидати: ['Logistic Regression', 'k-NN', 'GaussianNB', 'LDA', 'QDA', 'SVM']


### CV ROC-криві кандидатів

In [27]:
from sklearn.metrics import roc_curve, roc_auc_score

plt.figure(figsize=(7, 6))
for name, sc in cv_scores.items():
    fpr, tpr, _ = roc_curve(y_train, sc)
    plt.plot(fpr, tpr, label=name + " (AUC = " + str(round(roc_auc_score(y_train, sc), 3)) + ")")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("CV ROC-криві кандидатів (позавибіркові прогнози на train)")
plt.legend()
plt.grid(True)
plt.show()

**Що видно.** Усі шість кривих лежать вище діагоналі й близько одна до одної. Найвищі CV ROC AUC мають логістична регресія (0,9002), LDA (0,8993), k-NN (0,8971) і SVM (0,8968), нижче QDA (0,8848) і GaussianNB (0,8773). Тому за ROC AUC чотири кращі моделі практично не відрізняються. Ці криві побудовано за позавибірковими прогнозами на train. Остаточну незалежну криву будуємо окремо для фінальної моделі на test.

### CV-матриці помилок

In [28]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_models = ["Logistic Regression", "k-NN", prob_best_name, "SVM"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
rows = []
for ax, name in zip(axes, cm_models):
    pipe = candidates[name]
    pred = cross_val_predict(pipe, X_train, y_train, cv=cv)
    cm = confusion_matrix(y_train, pred)
    ConfusionMatrixDisplay(cm, display_labels=["0", "1"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
    tn, fp, fn, tp = cm.ravel()
    rows.append({"Модель": name, "Правильних": tn + tp, "FP": fp, "FN": fn, "TP": tp, "TN": tn, "Recall": tp / (tp + fn)})
plt.tight_layout()
plt.show()
display(pd.DataFrame(rows).set_index("Модель").round(4))
print("Усього об'єктів у train:", len(y_train))

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logi

,Правильних,FP,FN,TP,TN,Recall
Модель,,,,,,
Logistic Regression,207,13,22,89,118,0.8018
k-NN,205,17,20,91,114,0.8198
LDA,203,13,26,85,118,0.7658
SVM,206,12,24,87,119,0.7838


Усього об'єктів у train: 242


**Що видно (train, 242 об'єкти, позавибіркові прогнози).**

- Логістична регресія: правильних 207, FP = 13, FN = 22, Recall 0,8018.
- k-NN: правильних 205, FP = 17, FN = 20, Recall 0,8198.
- LDA: правильних 203, FP = 13, FN = 26, Recall 0,7658.
- SVM: правильних 206, FP = 12, FN = 24, Recall 0,7838.

Кількість правильних прогнозів майже однакова (203 до 207), але структура помилок різна. k-NN пропускає найменше хворих (20), зате найчастіше помилково тривожить здорових (17). LDA пропускає найбільше хворих (26). Якщо головний ризик це пропустити хворого, k-NN трохи привабливіша, а якщо важливі лише загальні помилки, різниці між моделями майже немає.

### Єдина таблиця CV-порівняння

In [29]:
rows = []
for name, pipe in candidates.items():
    row, _ = run_cv(pipe)
    rows.append({
        "Модель": name,
        "Найкращі параметри": candidate_params[name],
        "CV F1": row["CV F1"],
        "CV F1 std": row["F1 std"],
        "CV Recall": row["Recall"],
        "CV ROC AUC": row["ROC AUC"],
        "Час навчання, с": row["Час навчання, с"]
    })

df_cv_comparison = pd.DataFrame(rows)
display(df_cv_comparison.round(4))

CV_COMPARISON_PATH = os.path.join(REPORTS_DIR, "cv_model_comparison.csv")
df_cv_comparison.to_csv(CV_COMPARISON_PATH, index=False)
print("Таблицю збережено:", CV_COMPARISON_PATH)

,Модель,Найкращі параметри,CV F1,CV F1 std,CV Recall,CV ROC AUC,"Час навчання, с"
0,Logistic Regression,"penalty=l2, C=10.0, solver=liblinear",0.8363,0.0127,0.8016,0.9002,0.0117
1,k-NN,"MinMaxScaler, k=11, p=1.0",0.8291,0.0375,0.8190,0.8971,0.0103
2,GaussianNB,-,0.7495,0.0215,0.8458,0.8773,0.0106
3,LDA,-,0.8128,0.0257,0.7656,0.8993,0.0110
4,QDA,reg_param=0.7,0.8022,0.0451,0.7561,0.8848,0.0107
5,SVM,"kernel=rbf, C=10.0, gamma=0.01",0.8270,0.0330,0.7830,0.8968,0.0111


Таблицю збережено: /content/drive/MyDrive/3_course/Image_Recognition_and_Classical_Machine_Learning/reports/cv_model_comparison.csv


### Правило вибору фінальної моделі

Фінальну модель обираю лише за CV на train, test поки не відкривав. Правило задаю до перегляду будь-яких test-результатів:

1. Основний критерій це найбільший середній CV F1.
2. Усі моделі, чий CV F1 не нижчий за (найкращий F1 мінус його std), вважаю статистично не відмінними від найкращої.
3. Серед них беру модель з найбільшим CV Recall, бо за постановкою задачі пропустити хворого гірше, ніж зайвий раз направити здорового. При однаковому Recall беру меншу std.
4. Складність налаштування, швидкість і інтерпретація тут другорядні (усі моделі навчаються за соті частки секунди), їх згадую в обґрунтуванні.

In [30]:
best_row = df_cv_comparison.loc[df_cv_comparison["CV F1"].idxmax()]
threshold = best_row["CV F1"] - best_row["CV F1 std"]
tied = df_cv_comparison[df_cv_comparison["CV F1"] >= threshold]
print("Найкращий CV F1:", best_row["Модель"], round(best_row["CV F1"], 4), "| std:", round(best_row["CV F1 std"], 4))
print("Поріг (F1 мінус std):", round(threshold, 4))
print("Моделі в межах порога:")
display(tied[["Модель", "CV F1", "CV F1 std", "CV Recall", "CV ROC AUC"]].round(4))

final_row = tied.sort_values(["CV Recall", "CV F1 std"], ascending=[False, True]).iloc[0]
final_name = final_row["Модель"]
print("Фінальна модель за правилом:", final_name)

Найкращий CV F1: Logistic Regression 0.8363 | std: 0.0127
Поріг (F1 мінус std): 0.8236
Моделі в межах порога:


,Модель,CV F1,CV F1 std,CV Recall,CV ROC AUC
0,Logistic Regression,0.8363,0.0127,0.8016,0.9002
1,k-NN,0.8291,0.0375,0.8190,0.8971
5,SVM,0.8270,0.0330,0.7830,0.8968


Фінальна модель за правилом: k-NN


### Обґрунтування фінальної моделі

Найвищий CV F1 має логістична регресія: 0,8363 при std 0,0127. Поріг для «не відмінних» моделей дорівнює 0,8236. У нього потрапили k-NN (0,8291) і SVM (0,8270). Серед трьох найбільший CV Recall у k-NN (0,8190 проти 0,8016 у логістичної регресії та 0,7830 у SVM), тому за правилом фінальною обрано **k-NN** (MinMaxScaler, p = 1, k = 11). Її CV ROC AUC 0,8971 майже такий самий, як у логістичної регресії (0,9002). Перевага k-NN в найбільшому Recall і простоті ідеї, а недоліки в тому, що std (0,0375) утричі більший, ніж у логістичної регресії, прогноз повільніший (модель зберігає весь train), а інтерпретація гірша, ніж у коефіцієнтів. До того ж k-NN було вибрано з 60 конфігурацій, тому її CV F1 міг трохи завищитися. За сукупністю критеріїв, насамперед за Recall, модель зафіксовано як фінальну до оцінювання на test.

### Зафіксована фінальна модель

- **Алгоритм:** `KNeighborsClassifier` у складі Pipeline з усією попередньою обробкою.
- **Гіперпараметри:** `n_neighbors = 11`, `metric = "minkowski"`, `p = 1` (мангеттенська відстань), ваги за замовчуванням.
- **Числові ознаки** (age, trestbps, chol, thalach, oldpeak): заповнення пропусків медіаною та `MinMaxScaler`.
- **Категоріальні ознаки** (sex, cp, fbs, restecg, exang, slope, ca, thal): заповнення найчастішим значенням та `OneHotEncoder(handle_unknown="ignore")`.
- **Навчання:** на всій X_train, y_train. Поділ 80/20, `stratify = y`, `random_state = 42`. Позитивний клас 1.

Після цього пункту модель не змінюється.

### Одноразове оцінювання фінальної моделі на test

In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

final_pipeline = clone(candidates[final_name])
final_pipeline.fit(X_train, y_train)

y_pred = final_pipeline.predict(X_test)
if hasattr(final_pipeline, "predict_proba"):
    y_score = final_pipeline.predict_proba(X_test)[:, 1]
else:
    y_score = final_pipeline.decision_function(X_test)

df_final_test_metrics = pd.DataFrame([{
    "Фінальна модель": final_name,
    "Test Accuracy": accuracy_score(y_test, y_pred),
    "Test Precision": precision_score(y_test, y_pred),
    "Test Recall": recall_score(y_test, y_pred),
    "Test F1": f1_score(y_test, y_pred),
    "Test ROC AUC": roc_auc_score(y_test, y_score)
}])
display(df_final_test_metrics.round(4))

FINAL_METRICS_PATH = os.path.join(REPORTS_DIR, "final_test_metrics.csv")
df_final_test_metrics.to_csv(FINAL_METRICS_PATH, index=False)
print("Test-метрики збережено:", FINAL_METRICS_PATH)

fpr, tpr, _ = roc_curve(y_test, y_score)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label="AUC = " + str(round(roc_auc_score(y_test, y_score), 4)))
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-крива фінальної моделі на test")
plt.legend()
plt.grid(True)
plt.show()

cm_test = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm_test, display_labels=["0", "1"]).plot(colorbar=False)
plt.title("Матриця помилок фінальної моделі на test")
plt.show()
tn, fp, fn, tp = cm_test.ravel()
print("TN =", tn, "| FP =", fp, "| FN =", fn, "| TP =", tp)
print("Розмір test:", len(y_test), "| клас 0:", (y_test == 0).sum(), "| клас 1:", (y_test == 1).sum())

,Фінальна модель,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC AUC
0,k-NN,0.8689,0.8333,0.8929,0.8621,0.9524


Test-метрики збережено: /content/drive/MyDrive/3_course/Image_Recognition_and_Classical_Machine_Learning/reports/final_test_metrics.csv
TN = 28 | FP = 5 | FN = 3 | TP = 25
Розмір test: 61 | клас 0: 33 | клас 1: 28


**Що видно.** Фінальну k-NN навчено на всій X_train і один раз перевірено на 61 test-об'єкті (33 здорових, 28 хворих). Accuracy 0,8689, Precision 0,8333, Recall 0,8929, F1 0,8621, ROC AUC 0,9524. За матрицею помилок: правильних 53 із 61 (28 здорових і 25 хворих), 5 здорових помилково визнано хворими (FP), 3 хворих пропущено (FN).

**Порівняння з CV.** Test не гірший за крос-валідацію: F1 0,8621 проти 0,8291, Recall 0,8929 проти 0,8190, ROC AUC 0,9524 проти 0,8971. Різниця F1 (0,033) менша за std між фолдами (0,0375), тому це добре узгоджується з CV. Але test дуже малий: один пропущений хворий змінює Recall приблизно на 0,036, а одна помилка змінює accuracy на 0,016. Тож це не доказ, що на нових даних модель буде не гіршою, а лише відсутність ознак перенавчання. Після перегляду test-метрик модель я не змінював.

### Аналіз помилкових прогнозів

In [32]:
test_results = X_test.copy()
test_results["y_true"] = y_test
test_results["y_pred"] = y_pred
test_results["score"] = y_score
test_results["тип помилки"] = np.where(
    (test_results["y_true"] == 0) & (test_results["y_pred"] == 1), "FP",
    np.where((test_results["y_true"] == 1) & (test_results["y_pred"] == 0), "FN", "ok")
)

mistakes = test_results[test_results["тип помилки"] != "ok"]
print("Помилкових прогнозів:", len(mistakes), "із", len(test_results))
display(mistakes.round(3))

threshold_score = 0.5 if hasattr(final_pipeline, "predict_proba") else 0.0
print("Поріг рішення:", threshold_score)
print("Модуль відстані до порога для помилок:")
print((mistakes["score"] - threshold_score).abs().round(3).sort_values().to_string())
print()
print("Середня впевненість правильних прогнозів:", round((test_results[test_results["тип помилки"] == "ok"]["score"] - threshold_score).abs().mean(), 3))

Помилкових прогнозів: 8 із 61


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,y_true,y_pred,score,тип помилки
271,66,1,4,160,228,0,2,138,0,2.3,1,0.0,6.0,0,1,0.636,FP
196,69,1,1,160,234,1,2,131,0,0.1,2,1.0,3.0,0,1,0.545,FP
6,62,0,4,140,268,0,2,160,0,3.6,3,2.0,3.0,1,0,0.364,FN
33,59,1,4,135,234,0,0,161,0,0.5,2,0.0,7.0,0,1,0.636,FP
231,55,0,4,180,327,0,1,117,1,3.4,2,0.0,3.0,1,0,0.455,FN
66,60,1,3,140,185,0,2,155,0,3.0,2,0.0,3.0,1,0,0.273,FN
92,62,1,3,130,231,0,0,146,0,1.8,2,3.0,7.0,0,1,0.818,FP
131,51,1,3,94,227,0,0,154,1,0.0,1,1.0,7.0,0,1,0.727,FP


Поріг рішення: 0.5
Модуль відстані до порога для помилок:
196    0.045
231    0.045
6      0.136
271    0.136
33     0.136
66     0.227
131    0.227
92     0.318

Середня впевненість правильних прогнозів: 0.344


**Що видно.** Модель помилилась на 8 test-об'єктах із 61: п'ять FP (індекси 271, 196, 33, 92, 131) і три FN (6, 231, 66).

- **Близькість до межі.** Два об'єкти лежать майже на порозі 0,5 (196 і 231, відстань 0,045, тобто 6 голосів проти 5 з 11 сусідів), ще три на відстані 0,136 (6, 271, 33). Для цих п'яти випадків модель «вагалась», і їх можна розглядати як пограничні. Водночас об'єкти 92 (0,818) і 131 (0,727) помилково отримали високу ймовірність класу 1: у 9 і 8 з 11 сусідів клас був 1, тож модель була впевнена і помилилась. Середня відстань до порога на правильних прогнозах 0,344, а на помилках лише близько 0,16, тобто на помилках модель загалом менш впевнена.
- **Повторювані ознаки.** У трьох із п'яти FP `thal = 7` (об'єкти 33, 92, 131), а це значення в даних зазвичай пов'язане з класом 1, тож такі здорові пацієнти схожі на хворих. Усі три FN мають `thal = 3` (6, 231, 66), що типово для класу 0, тому хворі виглядають як здорові. Обидві групи помилок схожі на «винятки» за ознакою thal.
- **Відповідь на питання про надмірну впевненість.** Так, у двох випадках із восьми (92 і 131) модель була досить впевнена.

Цей аналіз пояснює результат, але не є підставою міняти модель.

### Збереження фінального Pipeline

In [33]:
import joblib

MODEL_PATH = os.path.join(MODELS_DIR, "best_classifier.joblib")
joblib.dump(final_pipeline, MODEL_PATH)
print("Файл моделі існує:", os.path.exists(MODEL_PATH))

loaded_pipeline = joblib.load(MODEL_PATH)
check_prediction = loaded_pipeline.predict(X_test.head(3))
print("Прогноз завантаженої моделі:", check_prediction)
print("Прогноз оригінальної моделі:", final_pipeline.predict(X_test.head(3)))
print("Усі прогнози на test збігаються:", bool((loaded_pipeline.predict(X_test) == y_pred).all()))

Файл моделі існує: True
Прогноз завантаженої моделі: [0 1 0]
Прогноз оригінальної моделі: [0 1 0]
Усі прогнози на test збігаються: True


## Підсумкові висновки

**Масштабування і логістична регресія.** CV F1 (0,8245) і Recall (0,7834) не змінились, ROC AUC трохи вищий зі StandardScaler (0,9025 проти 0,9001). Різниця в збіжності: 18 ітерацій проти 1371.

**Масштабування і k-NN.** Для k-NN масштаб критичний: без нього F1 0,5120, зі StandardScaler 0,7989, з MinMaxScaler 0,8008 (при k = 5).

**Найкращі k і відстань.** У схемі зі StandardScaler і k від 3 до 15 найкращим було k = 3 (F1 0,8042), мангеттенська відстань дала трохи більше (0,8175). У повному переборі найкращою стала конфігурація MinMaxScaler, p = 1, k = 11 з CV F1 0,8291. Усі різниці між топ-конфігураціями менші за розкид між фолдами.

**Найстабільніша ймовірнісна модель.** За F1 і ROC AUC найкраща LDA (0,8128 і 0,8993, std 0,0257). GaussianNB має найменший std (0,0215), але найнижчий F1 (0,7495) і найгірші ймовірності (Brier 0,2367).

**Стабільність QDA.** При reg_param = 0 QDA не працює (помилка рангу коваріаційної матриці). При reg_param від 0,1 до 0,7 вона працює прийнятно (F1 близько 0,80), але має більший розкид між фолдами (std до 0,045).

**Параметри SVM.** Обрано RBF-ядро з C = 10 і gamma = 0,01: CV F1 0,8270, ROC AUC 0,8968. Лінійна SVM поступилась (0,8025 при C = 1).

**Найкращі CV F1 і ROC AUC.** Обидві метрики найвищі у логістичної регресії (F1 0,8363, ROC AUC 0,9002).

**Фінальна модель.** k-NN (MinMaxScaler, p = 1, k = 11), бо за заздалегідь заданим правилом серед моделей, не відмінних за F1 від найкращої, вона має найбільший Recall (0,8190).

**Результати на test.** Accuracy 0,8689, Precision 0,8333, Recall 0,8929, F1 0,8621, ROC AUC 0,9524.

**Узгодженість із CV.** Test не гірший за CV (F1 0,8621 проти 0,8291), різниця в межах розкиду між фолдами.

**Обмеження.** Датасет малий (303 об'єкти, test лише 61), тому всі оцінки мають широку невизначеність: різниця між топ-моделями менша за розкид між фолдами. Конфігурацію k-NN вибрано з великого переліку на тих самих фолдах, тому її CV міг бути трохи завищеним. Сітка SVM має межу при C = 10. Модель навчальна й не придатна для клінічних рішень.

## Відповіді на контрольні запитання

1. **Для чого Pipeline?** Він складає всі кроки (пропуски, масштаб, кодування, модель) в один об'єкт, тому на кожному фолді все навчається лише на його train-частині й немає витоку.
2. **Яку проблему вирішує ColumnTransformer?** Дозволяє застосувати різну обробку до різних груп стовпців: числові масштабувати, категоріальні кодувати.
3. **Чому split до імпутації та масштабування?** Інакше медіана чи середнє рахуються і по test, тобто test «підглядає» в навчання й оцінка стає завищеною.
4. **Чому числові й категоріальні ознаки обробляються по-різному?** Числа мають порядок і величину, а коди категорій лише позначають групи, тому їх масштабувати безглуздо, їх треба розгортати у 0/1 колонки.
5. **Що означає handle_unknown="ignore"?** Якщо в test або під час використання з'явиться категорія, якої не було в train, енкодер не впаде з помилкою, а запише для неї нулі.
6. **Чому для GaussianNB, LDA і QDA зручна щільна матриця?** Ці моделі не працюють з розрідженими матрицями, тому `sparse_output=False` дає звичайний масив.
7. **Чому масштаб важливий для регресії, k-NN і SVM?** k-NN і SVM залежать від відстаней, тож ознаки з великим діапазоном домінують, а в регресії масштаб впливає на збіжність і на порівняння коефіцієнтів.
8. **Що контролює C в логістичній регресії?** Силу регуляризації навпаки: мале C дає сильний штраф і простішу модель, велике C слабкий штраф.
9. **Чим L1 відрізняється від L2?** L1 штрафує суму модулів і може обнуляти коефіцієнти, L2 штрафує суму квадратів і лише зменшує їх.
10. **Як k впливає на k-NN?** Мале k дає гнучку межу з ризиком перенавчання, велике згладжує межу й може призвести до недонавчання.
11. **Чому k-NN чутливий до масштабу?** Бо клас визначається за відстанню, а ознаки з великим діапазоном вносять у неї найбільше.
12. **Які припущення в Gaussian Naive Bayes?** Ознаки умовно незалежні в межах класу й мають нормальний розподіл.
13. **Чим LDA відрізняється від QDA?** LDA припускає однакову коваріаційну матрицю для всіх класів (лінійна межа), QDA дозволяє окрему матрицю для кожного класу (квадратична межа).
14. **Чому QDA може бути нестабільною?** Їй треба оцінити окрему коваріаційну матрицю для кожного класу, а при малій вибірці й корельованих ознаках вона вироджується.
15. **Що означає C у SVM?** Баланс між шириною розділювального відступу й штрафом за помилки: велике C сильніше карає помилки, мале допускає їх.
16. **Що означає gamma?** Радіус впливу одного об'єкта в RBF-ядрі: мала gamma дає гладку межу, велика дуже звивисту.
17. **Для чого GridSearchCV?** Автоматично перебирає комбінації гіперпараметрів і оцінює кожну за крос-валідацією.
18. **Що таке StratifiedKFold?** Крос-валідація, у якій кожен фолд зберігає таку саму частку класів, як увесь набір.
19. **Чому не підбирати параметри на тестовій вибірці?** Тоді test стає частиною навчання й перестає бути чесною перевіркою на нових даних.
20. **Чому не вибирати модель за Test F1 кількох моделей?** Це те саме, що підбір на test: найкращий результат частково випадковий, і оцінка стає завищеною.
21. **Що таке позавибірковий прогноз cross_val_predict?** Прогноз для кожного об'єкта від моделі, яка його не бачила під час навчання на відповідному фолді.
22. **Що показує ROC-крива?** Залежність частки знайдених хворих (TPR) від частки помилкових тривог (FPR) при різних порогах.
23. **Що означає ROC AUC?** Ймовірність того, що випадковий об'єкт класу 1 отримає вищу оцінку, ніж випадковий об'єкт класу 0. 0,5 це вгадування, 1 ідеал.
24. **Чому позитивний клас треба задавати явно?** Precision, Recall і F1 рахуються саме для нього, і без явного вибору метрики можуть бути про інший клас.
25. **Чому найкраща за accuracy модель не завжди найкраща?** Accuracy не розрізняє типи помилок: у медицині пропущений хворий дорожчий за хибну тривогу, і при дисбалансі класів accuracy взагалі може бути оманливою.
26. **Чому фінальну модель фіксують до відкриття test?** Щоб test лишився незалежною перевіркою, а вибір не залежав від його результатів.
27. **Що робити, якщо test гірший за CV?** Чесно описати як оцінку узагальнення й проаналізувати причину, але не підбирати іншу модель за цим test. Для нового вибору потрібен новий незалежний набір.
28. **Чому зберігати весь Pipeline?** Бо разом із моделлю зберігаються медіани, масштаб і кодування, і нові дані можна подати «сирими» без ручної обробки.
29. **Для чого окрема Git-гілка?** Щоб вести роботу ізольовано від main, мати чисту історію змін і не зламати стабільну версію.
30. **Для чого Pull Request?** Щоб описати зміни, переглянути їх (самому або викладачу) і після перевірки об'єднати гілку з main.